# Build the Science, Space, and Technology Sample

| Item | Description |
|---|---|
| Goal | Select the hearings of the two science committees out of the 36,378-package corpus and keep the usable ones. This is the demonstration sample the summarization stage runs on. The committees are selected by `authorityId`, which survives renames: `hssy00` covers the House Science Committee under all three of its names, and `sscm00` is the Senate Committee on Commerce, Science, and Transportation, where the Senate's science jurisdiction sits. |
| Input | `Processed/hearings_all.parquet` in the D1.1 scratch folder, from D1.1.4. |
| Output | `Processed/hearings_sst.parquet` in the D1.4 scratch folder: the selected and usable hearings with every column of the corpus plus `committee_scope`. |
| Variables | • `committee_scope`: `house_science`, `senate_commerce`, or `both`, assigned from every committee on the record, not only the first |
| Run | Last run 2026-08-20; the cell outputs shown are from that run. Not rerun: T1, T2, and T3 read this file. It now sits in the Processed folder named above. |

## Step 1. Import packages

In [ ]:
from pathlib import Path

import pandas as pd

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D1 - Data Collection and Processing/D1.4 - SST Sample Construction")
NOTEBOOK_NAME = "D1.4.1 - SST - Build SST Sample"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FOLDER = SCRATCH_FOLDER / "D1 - Data Collection and Processing/D1.1 - GovInfo Congressional Hearings" / "Processed"

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Set the committee identifiers and load the corpus

A keyword match on science, space, or technology would be wrong in both directions. It pulls in the Special Committee on the Year 2000 Technology Problem and misses a renamed title. The identifier also catches one Senate row mislabeled with the House id.

In [1]:
HOUSE_SCIENCE = "hssy00"  # Committee on Science; on Science and Technology; on Science, Space, and Technology
SENATE_COMMERCE = "sscm00"  # Committee on Commerce, Science, and Transportation
SST_IDS = {HOUSE_SCIENCE, SENATE_COMMERCE}

SOURCE = INPUT_FOLDER / "hearings_all.parquet"
TARGET = PROCESSED_FOLDER / "hearings_sst.parquet"

corpus = pd.read_parquet(SOURCE)
print(f"corpus : {len(corpus):,} packages, {len(corpus.columns)} columns")
print(f"usable : {int(corpus['usable'].sum()):,}")

corpus : 36,378 packages, 40 columns
usable : 33,324


## Step 4. Select the committees

`committee_scope` comes from the identifiers present on each row, so a hearing listing both committees is labeled `both`. The titles the two identifiers cover are printed. The reader can then see that the renames and the mislabeled Senate row are all caught.

In [2]:
in_house = pd.Series([HOUSE_SCIENCE in set(v) for v in corpus["committee_ids"]], index=corpus.index)
in_senate = pd.Series([SENATE_COMMERCE in set(v) for v in corpus["committee_ids"]], index=corpus.index)
selected = in_house | in_senate

scope = pd.Series(pd.NA, index=corpus.index, dtype="object")
scope[in_house] = "house_science"
scope[in_senate] = "senate_commerce"
scope[in_house & in_senate] = "both"
corpus["committee_scope"] = scope

print(f"House science   {HOUSE_SCIENCE}  {int(in_house.sum()):>6,}")
print(f"Senate commerce {SENATE_COMMERCE}  {int(in_senate.sum()):>6,}")
print(f"either                  {int(selected.sum()):>6,}")
print(f"both                    {int((in_house & in_senate).sum()):>6,}")
print()
covered = corpus.loc[selected, ["chamber", "committee_name"]].explode("committee_name")
covered = covered[covered["committee_name"].str.contains("Science|Commerce", na=False)]
print("committee titles the two identifiers cover:")
print(covered.value_counts().to_string())
print()
print("selected hearings by chamber:")
print(corpus.loc[selected, "chamber"].value_counts().to_string())

House science   hssy00     964
Senate commerce sscm00   1,420
either                   2,382
both                         2

committee titles the two identifiers cover:
chamber  committee_name                                    
SENATE   Committee on Commerce, Science, and Transportation    1414
HOUSE    Committee on Science, Space, and Technology            582
         Committee on Science and Technology                    216
         Committee on Science                                   139
JOINT    Committee on Science                                     3
HOUSE    Committee on Energy and Commerce                         2

selected hearings by chamber:
chamber
SENATE    1417
HOUSE      960
JOINT        5


## Step 5. Apply the usability filter

A row must pass both tests from D1.1.4: `has_text`, and more than 8,650 characters of parsed text. Both counts are printed, because the gap between them is what the filter costs.

In [3]:
sst = corpus.loc[selected].copy()
before = len(sst)
sst = sst.loc[sst["usable"]].copy()

dropped = corpus.loc[selected & ~corpus["usable"]]
print(f"selected by committee   {before:>6,}")
print(f"  usable                {len(sst):>6,}")
print(f"  dropped               {len(dropped):>6,}")
if len(dropped):
    reason = pd.Series("both", index=dropped.index)
    reason = reason.mask(dropped["has_text"] & (dropped["n_chars"] <= 8650), "length only")
    reason = reason.mask(~dropped["has_text"] & (dropped["n_chars"] > 8650), "has_text only")
    print("\n  why they were dropped:")
    print("   " + reason.value_counts().to_string().replace("\n", "\n   "))
    print(f"\n  their median character count: {dropped['n_chars'].median():,.0f}")
    print(f"  kept rows' median            : {sst['n_chars'].median():,.0f}")

selected by committee    2,382
  usable                 2,352
  dropped                   30

  why they were dropped:
   both             29
   has_text only     1

  their median character count: 1,036
  kept rows' median            : 232,003


## Step 6. Write the sample

Same row-group size as the corpus. The nested turn column cannot be read back if one group holds more than 2 GB.

In [4]:
sst = sst.sort_values(["year", "packageId"]).reset_index(drop=True)
sst.to_parquet(TARGET, compression="zstd", index=False, row_group_size=2000)

size = TARGET.stat().st_size
print(f"wrote {TARGET}")
print(f"  {len(sst):,} hearings, {size / 1e6:,.0f} MB zstd")
print(f"  {sst['n_words'].sum() / 1e6:,.0f} million words across {int(sst['n_turns'].sum()):,} turns")
print(f"  fiscal years {int(sst['year'].min())}-{int(sst['year'].max())}")
print()
print(sst["committee_scope"].value_counts().to_string())

wrote /project2/njia_1089/acroh/projects/llm_method/data/raw/02_hearings/hearings_sst.parquet
  2,352 hearings, 203 MB zstd
  94 million words across 399,041 turns
  fiscal years 1996-2025

committee_scope
senate_commerce    1411
house_science       939
both                  2


## Step 7. Check the sample

Four checks stop the notebook if they fail; three descriptions follow.

- One row per hearing.
- Every row carries a science committee and passes the usability filter.
- Every row has turns and clears the length threshold.
- Every qualifying hearing in the corpus is present.
- The shape of the sample by scope, by year, and by text structure, which is what the chunker will meet.

In [5]:
back = pd.read_parquet(TARGET)

assert len(back) == len(sst), f"{len(back):,} rows written against {len(sst):,}"
assert not back["packageId"].duplicated().any(), "duplicate packageId"
print(f"1) {len(back):,} hearings, packageId unique")

carries = pd.Series([bool(SST_IDS & set(v)) for v in back["committee_ids"]], index=back.index)
assert carries.all(), f"{int((~carries).sum())} rows carry neither science committee"
assert back["usable"].all(), "an unusable row entered the sample"
print("2) every row carries a science committee and passes the usability filter")

assert (back["n_turns"] > 0).all(), "a row has no turns"
assert (back["n_chars"] > 8650).all(), "a row is under the length threshold"
print(f"3) every row has turns and clears {8650:,} characters")

qualifies = pd.Series([bool(SST_IDS & set(v)) for v in corpus["committee_ids"]], index=corpus.index) & corpus["usable"]
expected = int(qualifies.sum())
assert len(back) == expected, f"{len(back):,} written, {expected:,} qualify"
print(f"4) all {expected:,} qualifying hearings are present")

print("\n5) shape of the sample:")
summary = back.groupby("committee_scope").agg(hearings=("packageId", "size"), median_words=("n_words", "median"),
                                              median_turns=("n_turns", "median"), first=("year", "min"), last=("year", "max"))
print(summary.to_string())

print("\n6) hearings per year:")
per_year = back.groupby("year").agg(hearings=("packageId", "size"), median_words=("n_words", "median"))
print(per_year.to_string())

print("\n7) how the text is structured, which is what the chunker will meet:")
print(f"   paragraph rule    : {back['paragraph_rule'].value_counts().to_dict()}")
print(f"   turns per hearing : median {back['n_turns'].median():,.0f}, p90 {back['n_turns'].quantile(0.9):,.0f}, max {int(back['n_turns'].max()):,}")
print(f"   words per hearing : median {back['n_words'].median():,.0f}, p90 {back['n_words'].quantile(0.9):,.0f}, max {int(back['n_words'].max()):,}")
single = int((back["n_turns"] <= 1).sum())
print(f"   hearings that came out as a single turn: {single:,}")
qfr = int((back["n_qfr_turns"] > 0).sum())
print(f"   hearings carrying questions for the record: {qfr:,}")

1) 2,352 hearings, packageId unique
2) every row carries a science committee and passes the usability filter
3) every row has turns and clears 8,650 characters
4) all 2,352 qualifying hearings are present

5) shape of the sample:
                 hearings  median_words  median_turns  first  last
committee_scope                                                   
both                    2       43641.5         148.5   2003  2003
house_science         939       22896.0         140.0   1996  2025
senate_commerce      1411       42007.0         154.0   1999  2025

6) hearings per year:
      hearings  median_words
year                        
1996         1       38156.0
1997         1       36236.0
1999        41       31560.0
2000        53       47393.0
2001        77       46247.0
2002        76       38056.5
2003       128       43942.0
2004        81       37144.0
2005        99       41410.0
2006       107       43035.0
2007       151       44724.0
2008        93       42772.0
2009  